### Stilvolle Päckchen – Lösung Schritt für Schritt

*42. Bundeswettbewerb Informatik (2023/2024), 2. Runde, Aufgabe 2*

Marius, Emily und Merle stört es, dass Bekleidung nur kurz getragen und anschließend weggeworfen wird. Sie finden heraus, dass große
Bekleidungsanbieter sogar Neuware wegwerfen, die in der aktuellen Saison nicht verkauft wurde. Gleichzeitig haben sie Werbung eines
Unternehmens entdeckt, welches komplette Outfits verkauft. Diese Idee möchten die drei nun auf ausgemusterte Kleidung anwenden.

Sie haben einen großen Bestand an Restware gekauft, um daraus Outfit-Boxen zusammenzustellen, die verschiedene Sorten (beispielsweise
Hose, Oberteil, Jacke, Schuhe, ...) kombinieren. Dabei achten sie darauf, dass die zusammengestellten Boxen nur Kleidungsstücke
enthalten, die von ihrem Stil her (zum Beispiel leger, sportlich, elegant, formell, ...) zusammen passen. Dafür haben die drei eine
Tabelle angelegt, die angibt, welche Stile kombinierbar sind. Zum Beispiel kann man legere mit sportlicher und auch mit eleganter
Kleidung kombinieren, aber nicht unbedingt sportliche mit eleganter.

Konkret soll für jede gepackte Box gelten:

a) Von jeder Sorte ist mindestens ein Kleidungsstück eingepackt, jedoch nicht mehr als drei.
b) Alle beteiligten Stile passen zusammen.

#### Aufgabe

Schreibe ein Programm, das den Nachwuchsmodezaren hilft. Es soll die Stil-Tabelle einlesen und für die eingekaufte Restware
ausgeben, wie die Boxen gepackt werden sollen, sodass möglichst wenige Kleidungsstücke übrig bleiben.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `paeckchen0.txt` bis `paeckchen7.txt`. Es gibt keine Formatbeschreibung; das Format haben wir aus den
Dateien abgelesen:

- Erste Zeile: Zahl der Sorten und Zahl der Stile, zum Beispiel `3 4`.
- Danach folgen Zeilen mit je zwei Stilen, die zusammenpassen, zum Beispiel `1 2`. Ein Stil passt immer zu sich selbst.
- Nach einer Leerzeile folgen Zeilen `Sorte Stil Anzahl`, zum Beispiel `1 2 55`: 55 Kleidungsstücke der Sorte 1 im Stil 2.

**Achtung:** `paeckchen2.txt` ist leer, auch in der Fassung auf der BWINF-Seite. Nach den Lösungshinweisen sind dort 3 Sorten,
9 Stile und 32 Kleidungsstücke gemeint; wir können diese Datei deshalb nicht auswerten und zeigen sie als „leer“.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**

- **Eine Aufgabe genauer ansehen, bevor man programmiert:** Aus „in jeder Box passen alle Stile zusammen“ folgt eine Struktur
  (Cliquen), die das Problem viel kleiner macht, als es zuerst aussieht.
- **Clique:** In einem Graphen ist eine Clique eine Menge von Knoten, die alle paarweise durch eine Kante verbunden sind. Hier sind
  die Knoten die Stile und die Kanten die Paare, die zusammenpassen. Eine **maximale** Clique lässt sich nicht mehr vergrößern.
- **Zufallsverfahren und Wiederholen:** Man probiert viele zufällige Lösungen und merkt sich die beste.
- **Lokale Suche (Bergsteigen):** Man startet mit irgendeiner Lösung, ändert sie ein wenig und behält die Änderung, wenn sie nicht
  schlechter ist. Bleibt man in einer Sackgasse stecken, hilft es, **manchmal auch eine Verschlechterung** zu erlauben
  (Simulated Annealing, „simuliertes Abkühlen“).
- **Ganzzahlige lineare Optimierung (ILP):** Ein Solver findet die beste Lösung und **beweist**, dass es keine bessere gibt.

**Nützliche Bibliotheken und Funktionen**

- `random.Random(1)` mit `choice` und `random` für Zufallszahlen, die bei jedem Lauf gleich sind
- `math.exp` für die Annahmewahrscheinlichkeit beim simulierten Abkühlen
- `min`, `sum`, Listen von Listen (Tabellen) und Wörterbücher (`dict`) mit Tupeln `(sorte, stil)` als Schlüssel

## Idee in Worten

1. **Einlesen und Prüfen.** Wir speichern die Tabelle `passt[a][b]` (passen die Stile `a` und `b` zusammen?) und die Tabelle
   `anzahl[sorte][stil]`. Eine Prüffunktion kontrolliert jede gepackte Lösung und zählt die übrigen Kleidungsstücke.
2. **Erster Versuch: Zufall.** Wie in den Lösungshinweisen: Erst viele „kleine“ Boxen mit genau einem Stück je Sorte zufällig packen,
   dann die Reste in vorhandene Boxen füllen. Das ist einfach, aber nicht gut.
3. **Die Struktur sehen.** In einer Box passen alle Stile paarweise zusammen (auch zwei Hosen!). Die Stile einer Box bilden also eine
   Clique. Jede Box lässt sich deshalb einer **maximalen Clique** zuordnen. Dann ist nur noch wichtig, **wie viele Stücke jeder Sorte**
   zu jeder maximalen Clique gehören. Eine Clique mit `b` Boxen braucht von jeder Sorte zwischen `b` und `3·b` Stücke, und umgekehrt
   lässt sich jede solche Menge auf `b` Boxen verteilen.
4. **Lokale Suche** über die Zuordnung „Kleidungsstück (Sorte, Stil) gehört zur Clique `k`“. Daraus bauen wir am Ende die Boxen.
5. **MiniZinc** beweist, dass diese Ergebnisse optimal sind.

## Schritt 1: Einlesen und Prüfen

`lies_paeckchen` liefert ein Wörterbuch mit der Zahl der Sorten, der Zahl der Stile, der Tabelle `passt` und der Tabelle `anzahl`.
Sorten und Stile zählen ab 1, deshalb haben die Tabellen eine unbenutzte Zeile und Spalte 0. Bei einer leeren Datei geben wir `None`
zurück.

Eine **Box** schreiben wir als Liste mit einem Eintrag pro Sorte; jeder Eintrag ist die Liste der Stile der Kleidungsstücke dieser
Sorte in der Box. Beispiel für drei Sorten: `[[1, 1], [2], [2, 1]]`. Die Funktion `pruefe_loesung` prüft Regel a) und b) und dass
nicht mehr Stücke verwendet werden, als da sind, und gibt die Zahl der übrigen Stücke zurück.

In [ ]:
def lies_paeckchen(dateiname):
    with open(dateiname, encoding='utf-8') as datei:
        text = datei.read().strip()
    if text == '':
        return None
    teile = text.split('\n\n')
    kopf = teile[0].split('\n')
    sorten, stile = [int(x) for x in kopf[0].split()]
    passt = []
    for a in range(stile + 1):
        passt.append([a == b for b in range(stile + 1)])      # ein Stil passt zu sich selbst
    for zeile in kopf[1:]:
        a, b = [int(x) for x in zeile.split()]
        passt[a][b] = True
        passt[b][a] = True
    anzahl = []
    for s in range(sorten + 1):
        anzahl.append([0] * (stile + 1))
    for zeile in teile[1].split('\n'):
        s, r, n = [int(x) for x in zeile.split()]
        anzahl[s][r] += n
    return {'sorten': sorten, 'stile': stile, 'passt': passt, 'anzahl': anzahl}


def gesamtzahl(p):
    return sum(sum(zeile) for zeile in p['anzahl'])


def pruefe_loesung(p, boxen):
    """Prüft alle Boxen und gibt die Zahl der übrigen Kleidungsstücke zurück (AssertionError bei einem Fehler)."""
    verbraucht = []
    for s in range(p['sorten'] + 1):
        verbraucht.append([0] * (p['stile'] + 1))
    for box in boxen:
        assert len(box) == p['sorten'], 'jede Sorte muss vorkommen'
        stile_der_box = []
        for s in range(p['sorten']):
            assert 1 <= len(box[s]) <= 3, 'je Sorte 1 bis 3 Stücke'
            for r in box[s]:
                verbraucht[s + 1][r] += 1
                stile_der_box.append(r)
        for a in stile_der_box:
            for b in stile_der_box:
                assert p['passt'][a][b], 'Stile %d und %d passen nicht' % (a, b)
    for s in range(1, p['sorten'] + 1):
        for r in range(1, p['stile'] + 1):
            assert verbraucht[s][r] <= p['anzahl'][s][r], 'zu viele Stücke (%d, %d)' % (s, r)
    return gesamtzahl(p) - sum(sum(zeile) for zeile in verbraucht)


NAMEN = ['paeckchen%d.txt' % i for i in range(8)]
daten = {}
for name in NAMEN:
    daten[name] = lies_paeckchen('beispieldaten/' + name)
    p = daten[name]
    if p is None:
        print(name, 'ist leer')
    else:
        print(name, p['sorten'], 'Sorten,', p['stile'], 'Stile,', gesamtzahl(p), 'Kleidungsstücke')

paeckchen0.txt 3 Sorten, 2 Stile, 11 Kleidungsstücke
paeckchen1.txt 3 Sorten, 4 Stile, 499 Kleidungsstücke
paeckchen2.txt ist leer
paeckchen3.txt 5 Sorten, 10 Stile, 96 Kleidungsstücke
paeckchen4.txt 7 Sorten, 24 Stile, 437 Kleidungsstücke
paeckchen5.txt 5 Sorten, 21 Stile, 174 Kleidungsstücke
paeckchen6.txt 5 Sorten, 10 Stile, 770 Kleidungsstücke
paeckchen7.txt 5 Sorten, 10 Stile, 700 Kleidungsstücke


## Schritt 2: Erster Versuch mit Zufall (Verfahren der Lösungshinweise)

Eine **minimale Box** hat genau ein Stück je Sorte, und die Stile passen paarweise zusammen. Wir suchen zuerst alle minimalen Boxen
(Rekursion über die Sorten). Dann:

1. Wähle zufällig eine minimale Box, für die noch genug Stücke da sind, packe sie, und wiederhole das, bis keine mehr geht.
2. Fülle die Reste auf: Wähle zufällig ein Stück und eine Box, in die es noch passt (weniger als drei Stücke der Sorte, alle Stile
   passen), und tue es hinein, bis nichts mehr geht.

Das ganze Verfahren wiederholen wir mehrmals mit anderen Zufallszahlen und behalten die beste Lösung. Hier nur **30 Läufe** je Datei,
damit das Notebook schnell bleibt.

In [ ]:
import random


def minimale_boxen(p):
    stile_je_sorte = []
    for s in range(1, p['sorten'] + 1):
        stile_je_sorte.append([r for r in range(1, p['stile'] + 1) if p['anzahl'][s][r] > 0])
    ergebnis = []

    def baue(s, box):
        if s == p['sorten']:
            ergebnis.append(box)
            return
        for r in stile_je_sorte[s]:
            if all(p['passt'][r][q] for q in box):
                baue(s + 1, box + [r])

    baue(0, [])
    return ergebnis


def zufaelliges_packen(p, minimale, zufall):
    rest = []
    for zeile in p['anzahl']:
        rest.append(zeile[:])

    def moeglich(box):
        for s in range(p['sorten']):
            if rest[s + 1][box[s]] == 0:
                return False
        return True

    kandidaten = [b for b in minimale if moeglich(b)]
    boxen = []
    while kandidaten:
        box = zufall.choice(kandidaten)
        boxen.append([[r] for r in box])
        for s in range(p['sorten']):
            rest[s + 1][box[s]] -= 1
        kandidaten = [b for b in kandidaten if moeglich(b)]
    while True:                                   # Reste auffüllen
        moeglichkeiten = []
        for s in range(p['sorten']):
            for r in range(1, p['stile'] + 1):
                if rest[s + 1][r] > 0:
                    for i in range(len(boxen)):
                        if len(boxen[i][s]) < 3 and all(p['passt'][r][q] for teil in boxen[i] for q in teil):
                            moeglichkeiten.append((s, r, i))
        if not moeglichkeiten:
            break
        s, r, i = zufall.choice(moeglichkeiten)
        boxen[i][s].append(r)
        rest[s + 1][r] -= 1
    return boxen


def bester_zufall(p, laeufe, zufall):
    minimale = minimale_boxen(p)
    beste = None
    for _ in range(laeufe):
        boxen = zufaelliges_packen(p, minimale, zufall)
        uebrig = pruefe_loesung(p, boxen)
        if beste is None or uebrig < beste[0]:
            beste = (uebrig, boxen)
    return beste


zufall_ergebnis = {}
for name in NAMEN:
    if daten[name] is not None:
        zufall_ergebnis[name] = bester_zufall(daten[name], 30, random.Random(1))
        print(name, 'übrig:', zufall_ergebnis[name][0], 'bei', len(zufall_ergebnis[name][1]), 'Boxen')

paeckchen0.txt übrig: 0 bei 3 Boxen
paeckchen1.txt übrig: 0 bei 134 Boxen
paeckchen3.txt übrig: 2 bei 12 Boxen
paeckchen4.txt übrig: 26 bei 31 Boxen
paeckchen5.txt übrig: 25 bei 17 Boxen
paeckchen6.txt übrig: 86 bei 87 Boxen
paeckchen7.txt übrig: 230 bei 42 Boxen


Bei `paeckchen0`, `1` und `3` ist das schon so gut wie bei den Lösungshinweisen (0, 0 und 2 übrig). Bei den größeren Beispielen ist es
deutlich schlechter als dort (26, 25, 86 und 230 übrig gegen 5, 19, 22 und 145 bei `paeckchen4` bis `7`; die Hinweise rechnen
90 Sekunden mit Verbesserungen und vielen Läufen). Mit mehr Läufen würde es besser, aber das lohnt sich nicht: Wir sehen gleich eine
bessere Idee.

## Schritt 3: Die Struktur sehen – Cliquen

In einer Box müssen **alle** Stile paarweise zusammenpassen (die Lösungshinweise legen fest, dass das auch für zwei Stücke derselben
Sorte gilt; wir machen es genauso). Die Stile in einer Box bilden also eine **Clique** im Graphen der Stile. Jede Clique steckt in
einer **maximalen** Clique. Wir ordnen jede Box einer maximalen Clique `K` zu, zu der alle ihre Stile gehören.

Dann zählt für `K` nur noch `n[s]`, die Zahl der Stücke der Sorte `s`, die in Boxen der Clique `K` liegen:

- **Notwendig:** Liegen in `b` Boxen jeweils 1 bis 3 Stücke der Sorte `s`, so ist `b ≤ n[s] ≤ 3·b`.
- **Hinreichend:** Gilt `b ≤ n[s] ≤ 3·b` für jede Sorte, so verteilt man die `n[s]` Stücke der Sorte reihum auf die `b` Boxen. Jede Box
  bekommt mindestens ein und höchstens drei Stücke, und alle Stile stammen aus `K`, passen also zusammen.

Eine Clique ist also **brauchbar**, wenn das **Größte** der `n[s]` höchstens das Dreifache des **Kleinsten** ist (mit `b` gleich dem
Kleinsten; ist das Kleinste 0, kann die Clique keine Box packen). Packen können wir dann `Σ min(n[s], 3·m)` Stücke mit `m = min(n[s])`,
überschüssige Stücke einer Sorte lassen wir übrig.

Mit dieser Sicht haben die Beispiele nur **wenige** maximale Cliquen, zwischen 1 und 11. Die Aufgabe heißt jetzt: **Ordne jedes
Kleidungsstück einer Clique zu (oder lass es übrig), sodass möglichst viele Stücke gepackt werden.** Die Lösungshinweise nennen für das
ILP über alle möglichen Boxen bis zu 2,7 Millionen Variablen; hier sind es ein paar Dutzend.

Zuerst die maximalen Cliquen (Bron-Kerbosch: Wir erweitern eine Clique `c` um Stile aus `kandidaten`, die zu allen Stilen in `c` passen;
`ausgeschlossen` enthält Stile, die schon in einer anderen gefundenen Clique stecken):

In [ ]:
def maximale_cliquen(p):
    alle = list(range(1, p['stile'] + 1))
    ergebnis = []

    def erweitere(clique, kandidaten, ausgeschlossen):
        if not kandidaten and not ausgeschlossen:
            ergebnis.append(clique)
            return
        for r in kandidaten[:]:
            nachbarn = [q for q in alle if q != r and p['passt'][r][q]]
            erweitere(clique + [r],
                      [q for q in kandidaten if q in nachbarn],
                      [q for q in ausgeschlossen if q in nachbarn])
            kandidaten.remove(r)
            ausgeschlossen.append(r)

    erweitere([], alle[:], [])
    return ergebnis


cliquen = {}
for name in NAMEN:
    if daten[name] is not None:
        cliquen[name] = maximale_cliquen(daten[name])
        print(name, len(cliquen[name]), 'maximale Cliquen')
print('paeckchen1:', cliquen['paeckchen1.txt'])
print('paeckchen6:', cliquen['paeckchen6.txt'])

paeckchen0.txt 1 maximale Cliquen
paeckchen1.txt 3 maximale Cliquen
paeckchen3.txt 6 maximale Cliquen
paeckchen4.txt 11 maximale Cliquen
paeckchen5.txt 8 maximale Cliquen
paeckchen6.txt 4 maximale Cliquen
paeckchen7.txt 4 maximale Cliquen
paeckchen1: [[1, 2], [2, 3], [3, 4]]
paeckchen6: [[1, 2, 3, 4, 10], [1, 3, 5, 6, 10], [1, 4, 8, 10], [7, 9, 10]]


## Schritt 4: Lokale Suche über die Zuordnung

Wir merken uns für jede (Sorte, Stil) und jede Clique `k`, wie viele Stücke dorthin zugeordnet sind (`zugeordnet[(s, r)][k]`).
Was nirgends zugeordnet ist, bleibt übrig. Die **Bewertung** einer Clique ist die Zahl der Stücke, die man dort packen kann:

```python
def wert_clique(n_je_sorte):
    m = min(n_je_sorte)
    return sum(min(n, 3 * m) for n in n_je_sorte)
```

Ein **Schritt**: Wähle zufällig ein (Sorte, Stil) und ein Stück davon, nimm es aus seiner jetzigen Clique (oder aus „übrig“) und lege
es in eine zufällige andere Clique, zu der der Stil gehört, oder in „übrig“. Berechne, wie sich die Bewertung der beiden betroffenen
Cliquen ändert (`d`).

- **Bergsteigen** (`temperatur = 0`): behalte den Schritt, wenn `d >= 0` (auch gleich gute Schritte, sonst käme man über
  Plateaus nie hinweg).
- **Simuliertes Abkühlen** (`temperatur > 0`): eine Verschlechterung `d < 0` behalten wir mit Wahrscheinlichkeit `exp(d / temperatur)`.
  Kleine Verschlechterungen sind also recht wahrscheinlich, große sehr selten.

Die beste Zuordnung, die wir unterwegs gesehen haben, merken wir uns. Zu Beginn wird jedes Stück zufällig einer passenden Clique
zugeordnet.

In [ ]:
import math


def wert_clique(n_je_sorte):
    m = min(n_je_sorte)
    return sum(min(n, 3 * m) for n in n_je_sorte)


def lokale_suche(p, cliquen_liste, schritte, zufall, temperatur):
    """Gibt (gepackte Stücke, Zuordnung) der besten Zuordnung zurück."""
    anzahl_k = len(cliquen_liste)
    stuecke = []                       # alle (Sorte, Stil) mit mindestens einem Stück
    for s in range(1, p['sorten'] + 1):
        for r in range(1, p['stile'] + 1):
            if p['anzahl'][s][r] > 0:
                stuecke.append((s, r))
    ziele = {}                         # mögliche Ziele: Cliquen mit dem Stil, und -1 für "übrig"
    for s, r in stuecke:
        ziele[(s, r)] = [k for k in range(anzahl_k) if r in cliquen_liste[k]] + [-1]
    n = []                             # n[k][s-1] = Stücke der Sorte s in Clique k
    for k in range(anzahl_k):
        n.append([0] * p['sorten'])
    zugeordnet = {}
    for s, r in stuecke:
        zugeordnet[(s, r)] = [0] * anzahl_k
        for _ in range(p['anzahl'][s][r]):
            k = zufall.choice(ziele[(s, r)][:-1])
            zugeordnet[(s, r)][k] += 1
            n[k][s - 1] += 1
    werte = [wert_clique(zeile) for zeile in n]
    summe = sum(werte)
    beste_summe = summe
    beste_zuordnung = {schluessel: zeile[:] for schluessel, zeile in zugeordnet.items()}
    for _ in range(schritte):
        s, r = zufall.choice(stuecke)
        quellen = [k for k in range(anzahl_k) if zugeordnet[(s, r)][k] > 0]
        if sum(zugeordnet[(s, r)]) < p['anzahl'][s][r]:
            quellen.append(-1)
        von = zufall.choice(quellen)
        nach = zufall.choice(ziele[(s, r)])
        if von == nach:
            continue
        d = 0
        if von >= 0:
            n[von][s - 1] -= 1
            neu_von = wert_clique(n[von])
            d += neu_von - werte[von]
        if nach >= 0:
            n[nach][s - 1] += 1
            neu_nach = wert_clique(n[nach])
            d += neu_nach - werte[nach]
        behalten = d >= 0 or (temperatur > 0 and zufall.random() < math.exp(d / temperatur))
        if behalten:
            if von >= 0:
                werte[von] = neu_von
                zugeordnet[(s, r)][von] -= 1
            if nach >= 0:
                werte[nach] = neu_nach
                zugeordnet[(s, r)][nach] += 1
            summe += d
            if summe > beste_summe:
                beste_summe = summe
                beste_zuordnung = {schluessel: zeile[:] for schluessel, zeile in zugeordnet.items()}
        else:                          # zurücknehmen
            if von >= 0:
                n[von][s - 1] += 1
            if nach >= 0:
                n[nach][s - 1] -= 1
    return beste_summe, beste_zuordnung


print('Beispiel paeckchen6, 20000 Schritte, Bergsteigen:')
gepackt, _ = lokale_suche(daten['paeckchen6.txt'], cliquen['paeckchen6.txt'], 20000, random.Random(1), 0)
print('übrig:', gesamtzahl(daten['paeckchen6.txt']) - gepackt)

Beispiel paeckchen6, 20000 Schritte, Bergsteigen:
übrig: 22


## Schritt 5: Aus der Zuordnung Boxen bauen

Für jede Clique `k` ist `m` das Kleinste der `n[s]`. Ist `m = 0`, entsteht keine Box. Sonst packen wir `m` Boxen: Von jeder Sorte nehmen
wir höchstens `3·m` Stücke und verteilen sie **reihum** auf die `m` Boxen (Stück Nummer `j` kommt in Box `j mod m`). Weil es mindestens
`m` Stücke sind, bekommt jede Box eins; weil es höchstens `3·m` sind, bekommt keine mehr als drei.

Die Funktion `pruefe_loesung` aus Schritt 1 kontrolliert das Ergebnis. Dann starten wir die lokale Suche für alle Beispiele, zuerst nur
mit Bergsteigen, dann mit simuliertem Abkühlen (`temperatur = 0.3`). Beide Male 200 000 Schritte.

In [ ]:
def baue_boxen(p, cliquen_liste, zuordnung):
    boxen = []
    for k in range(len(cliquen_liste)):
        liste_je_sorte = []            # die Stile aller Stücke der Sorte s in Clique k
        for s in range(1, p['sorten'] + 1):
            stile_der_sorte = []
            for r in range(1, p['stile'] + 1):
                if (s, r) in zuordnung:
                    stile_der_sorte += [r] * zuordnung[(s, r)][k]
            liste_je_sorte.append(stile_der_sorte)
        m = min(len(liste) for liste in liste_je_sorte)
        if m == 0:
            continue
        neue_boxen = [[[] for _ in range(p['sorten'])] for _ in range(m)]
        for s in range(p['sorten']):
            for j, r in enumerate(liste_je_sorte[s][:3 * m]):
                neue_boxen[j % m][s].append(r)
        boxen += neue_boxen
    return boxen


def suche_und_baue(p, cliquen_liste, schritte, zufall, temperatur):
    gepackt, zuordnung = lokale_suche(p, cliquen_liste, schritte, zufall, temperatur)
    boxen = baue_boxen(p, cliquen_liste, zuordnung)
    uebrig = pruefe_loesung(p, boxen)
    assert uebrig == gesamtzahl(p) - gepackt
    return uebrig, boxen


bergsteigen = {}
abkuehlen = {}
for name in NAMEN:
    if daten[name] is not None:
        bergsteigen[name] = suche_und_baue(daten[name], cliquen[name], 200000, random.Random(2), 0)
        abkuehlen[name] = suche_und_baue(daten[name], cliquen[name], 200000, random.Random(2), 0.3)
        print(name, 'übrig: Bergsteigen', bergsteigen[name][0], ', Abkühlen', abkuehlen[name][0],
              ', Boxen', len(abkuehlen[name][1]))

paeckchen0.txt übrig: Bergsteigen 0 , Abkühlen 0 , Boxen 3
paeckchen1.txt übrig: Bergsteigen 0 , Abkühlen 0 , Boxen 130
paeckchen3.txt übrig: Bergsteigen 2 , Abkühlen 2 , Boxen 11
paeckchen4.txt übrig: Bergsteigen 0 , Abkühlen 0 , Boxen 34
paeckchen5.txt übrig: Bergsteigen 19 , Abkühlen 19 , Boxen 18
paeckchen6.txt übrig: Bergsteigen 22 , Abkühlen 22 , Boxen 87
paeckchen7.txt übrig: Bergsteigen 131 , Abkühlen 131 , Boxen 50


Bei diesen Beispielen findet schon das reine Bergsteigen dieselben Werte wie das simulierte Abkühlen (bei `paeckchen5` und `7`
probiert mit mehreren Startwerten der Zufallszahlen, jedes Mal gleich). Die Werte stimmen mit den ILP-Werten der Lösungshinweise überein (19, 22, 131
bei `paeckchen5`, `6`, `7`, und 0 bei `paeckchen4`, das die Hinweise erst nach etwa 6 Stunden mit einem ILP-Solver lösen konnten).
Die Hinweise nennen für `paeckchen3` 2 übrige Stücke mit ihrem Zufallsverfahren, wir finden ebenfalls 2.

Ob 2 bei `paeckchen3` (und die anderen Werte) wirklich nicht zu unterbieten sind, weiß die lokale Suche nicht. Das beweisen wir im
letzten Abschnitt mit MiniZinc.

### Beispiele zur Ansicht

`paeckchen0` ist klein genug, um alle Boxen zu zeigen (Stile je Sorte):

In [ ]:
def zeige_boxen(boxen):
    for i, box in enumerate(boxen):
        teile = []
        for s, stile_der_sorte in enumerate(box):
            teile.append('Sorte %d: %s' % (s + 1, sorted(stile_der_sorte)))
        print('Box %2d:' % (i + 1), '; '.join(teile))


uebrig0, boxen0 = abkuehlen['paeckchen0.txt']
print('paeckchen0: übrig', uebrig0)
zeige_boxen(boxen0)

paeckchen0: übrig 0
Box  1: Sorte 1: [1, 2]; Sorte 2: [2]; Sorte 3: [2]
Box  2: Sorte 1: [1, 2]; Sorte 2: [2]; Sorte 3: [2]
Box  3: Sorte 1: [1]; Sorte 2: [2]; Sorte 3: [2]


Bei `paeckchen6` sieht man, wie die Boxen auf die vier maximalen Cliquen verteilt sind. Zu jeder Clique steht, aus welchen Stilen
sie besteht, wie viele Boxen daraus entstehen und wie viele Stücke je Sorte dort liegen (Summe über alle Boxen der Clique):

In [ ]:
def zeige_verteilung(cliquen_liste, boxen):
    zaehler = [0] * len(cliquen_liste)
    for box in boxen:
        for k in range(len(cliquen_liste)):             # erste Clique, die alle Stile der Box enthält
            if all(r in cliquen_liste[k] for teil in box for r in teil):
                zaehler[k] += 1
                break
    print('Clique (Stile)      Boxen')
    for k in range(len(cliquen_liste)):
        print('%-19s %d' % (cliquen_liste[k], zaehler[k]))


uebrig6, boxen6 = abkuehlen['paeckchen6.txt']
zeige_verteilung(cliquen['paeckchen6.txt'], boxen6)
print('Boxen insgesamt:', len(boxen6), ', übrig:', uebrig6)

Clique (Stile)      Boxen
[1, 2, 3, 4, 10]    17
[1, 3, 5, 6, 10]    34
[1, 4, 8, 10]       23
[7, 9, 10]          13
Boxen insgesamt: 87 , übrig: 22


## Zum Schluss: Optimal mit MiniZinc

Die Lösungshinweise beschreiben ein ILP mit einer Variablen je **Boxart** (Millionen bei `paeckchen3` und `paeckchen4`). Mit der
Sicht aus Schritt 3 geht es mit viel weniger: Wir wählen je Clique `k` die Zahl der Boxen `boxen[k]` und die Zahl `x[k, s, r]` der
Stücke von Sorte `s` und Stil `r`, die in Boxen der Clique `k` liegen. Bedingungen (alle linear):

- `x[k, s, r] = 0`, wenn der Stil `r` nicht zur Clique `k` gehört,
- je Clique und Sorte: `boxen[k] ≤ Σr x[k, s, r] ≤ 3 · boxen[k]`,
- je Sorte und Stil: `Σk x[k, s, r] ≤ anzahl[s, r]`.

Maximiert wird die Zahl der gepackten Stücke `Σ x`. Das ist genau die Zuordnung aus Schritt 4, nur dass der Solver sie **beweisbar
optimal** wählt. Neu gegenüber den Arbeitsblättern ist höchstens das Array mit drei Indizes `x[k, s, r]`.

```minizinc
int: S;                                   % Sorten
int: R;                                   % Stile
int: K;                                   % maximale Cliquen
array[1..K, 1..R] of 0..1: in_clique;     % in_clique[k, r] = 1, wenn Stil r zur Clique k gehört
array[1..S, 1..R] of int: anzahl;         % Kleidungsstücke je Sorte und Stil

array[1..K] of var 0..sum(anzahl): boxen;                 % Boxen je Clique
array[1..K, 1..S, 1..R] of var 0..max(anzahl): x;         % x[k, s, r]: Stücke (s, r) in Boxen der Clique k

% Stile außerhalb der Clique dürfen nicht in ihre Boxen
constraint forall(k in 1..K, s in 1..S, r in 1..R where in_clique[k, r] == 0)(x[k, s, r] = 0);
% Jede Box hat von jeder Sorte 1 bis 3 Stücke
constraint forall(k in 1..K, s in 1..S)(
  sum(r in 1..R)(x[k, s, r]) >= boxen[k] /\ sum(r in 1..R)(x[k, s, r]) <= 3 * boxen[k]);
% Es gibt nicht mehr Stücke, als vorhanden sind
constraint forall(s in 1..S, r in 1..R)(sum(k in 1..K)(x[k, s, r]) <= anzahl[s, r]);

solve maximize sum(x);

output ["uebrig = ", show(sum(anzahl) - sum(x)), "\n"];
```

Die Daten (Zahl der Sorten und Stile, die Cliquen aus Schritt 3 und die Anzahlen) baut die Python-Zelle `als_dzn`:

In [ ]:
def als_dzn(p, cliquen_liste):
    in_clique = []
    for clique in cliquen_liste:
        for r in range(1, p['stile'] + 1):
            in_clique.append('1' if r in clique else '0')
    anzahlen = []
    for s in range(1, p['sorten'] + 1):
        for r in range(1, p['stile'] + 1):
            anzahlen.append(str(p['anzahl'][s][r]))
    return 'S = %d;\nR = %d;\nK = %d;\nin_clique = array2d(1..%d, 1..%d, [%s]);\nanzahl = array2d(1..%d, 1..%d, [%s]);' % (
        p['sorten'], p['stile'], len(cliquen_liste), len(cliquen_liste), p['stile'], ', '.join(in_clique),
        p['sorten'], p['stile'], ', '.join(anzahlen))


print(als_dzn(daten['paeckchen0.txt'], cliquen['paeckchen0.txt']))

S = 3;
R = 2;
K = 1;
in_clique = array2d(1..1, 1..2, [1, 1]);
anzahl = array2d(1..3, 1..2, [3, 2, 0, 3, 0, 3]);


Mit dem Solver **HiGHS** (ein Löser für gemischt-ganzzahlige lineare Programme, in MiniZinc ausgewählt über *Solver configuration*) ist
die Suche bei allen sieben nicht leeren Beispielen sofort fertig. Für `paeckchen4`:

**Ausgabe:**

```
uebrig = 0
----------
==========
```

und für `paeckchen7`:

**Ausgabe:**

```
uebrig = 131
----------
==========
```

Beim Erzeugen dieses Notebooks haben wir das Modell auf alle sieben Beispiele angewendet: Der Solver meldet bei jedem „Suche beendet“
(`==========`) und genau die Zahl der übrigen Kleidungsstücke, die auch unsere lokale Suche mit simuliertem Abkühlen gefunden hat:
**0, 0, 2, 0, 19, 22, 131** für `paeckchen0`, `1`, `3`, `4`, `5`, `6`, `7`. Damit sind alle diese Werte **optimal**. Insbesondere
sind die Werte für `paeckchen3` (2 übrig) und `paeckchen4` (0 übrig), bei denen das ILP der Hinweise in annehmbarer Zeit nicht fertig
wurde, mit dem Cliquen-Modell kein Problem. Für `paeckchen7` bleiben mindestens 131 Stücke übrig; die 145 des Zufallsverfahrens in den
Hinweisen sind also nicht optimal.

**Grenzen anderer Solver:** Die Constraint-Solver **Gecode** und **Chuffed** werden mit demselben Modell innerhalb einer Minute nur
bei wenigen Beispielen fertig (Gecode nur bei `paeckchen0`, Chuffed bei `paeckchen0` und `paeckchen5`). Bei den anderen läuft das
Zeitlimit ab, und die bis dahin gefundenen Lösungen sind teils weit vom Optimum entfernt (Gecode bei `paeckchen1`: 109 statt 0 übrig).
Dieses Problem ist ein reines Summen- und Ungleichungsproblem, und dafür ist ein ILP-Solver wie HiGHS (oder Cbc) die richtige Wahl.

## Alle Ergebnisse im Vergleich

Übrige Kleidungsstücke. Spalten: Zufall (30 Läufe), Bergsteigen, Abkühlen (je 200 000 Schritte), die Werte der Lösungshinweise
(Tabelle 5, „Algo“ und „ILP“; – heißt: kein Wert angegeben) und das Optimum laut MiniZinc. Bei `paeckchen2` steht in Klammern, was
die Hinweise über die (leere) Datei sagen.

| Datei | Sorten | Stile | Stücke | Zufall | Bergsteigen | Abkühlen | Hinweise Algo | Hinweise ILP | optimal |
|---|---|---|---|---|---|---|---|---|---|
| paeckchen0 | 3 | 2 | 11 | 0 | 0 | 0 | 0 | 0 | 0 |
| paeckchen1 | 3 | 4 | 499 | 0 | 0 | 0 | 0 | 0 | 0 |
| paeckchen2 | (3) | (9) | (32) | leer | leer | leer | 0 | 0 | – |
| paeckchen3 | 5 | 10 | 96 | 2 | 2 | 2 | 2 | – | 2 |
| paeckchen4 | 7 | 24 | 437 | 26 | 0 | 0 | 5 | – | 0 |
| paeckchen5 | 5 | 21 | 174 | 25 | 19 | 19 | 19 | 19 | 19 |
| paeckchen6 | 5 | 10 | 770 | 86 | 22 | 22 | 22 | 22 | 22 |
| paeckchen7 | 5 | 10 | 700 | 230 | 131 | 131 | 145 | 131 | 131 |

Bergsteigen und Abkühlen erreichen bei allen Beispielen das Optimum; die Temperatur ist hier nicht nötig, bei schwierigeren
Daten könnte sie helfen, aus Sackgassen herauszukommen.

## Für Neugierige

- **Warum ist ein Beweis möglich?** Das Modell über maximale Cliquen ist **exakt** (Schritt 3), nicht nur eine Näherung: Jede Packung
  ergibt eine Lösung des Modells mit gleich vielen gepackten Stücken, und jede Lösung des Modells lässt sich mit `baue_boxen` in echte
  Boxen verwandeln. Deshalb ist der Optimalwert des Modells der Optimalwert der Aufgabe.
- **Rechnen ohne Solver:** Für eine feste Zahl `boxen[k]` je Clique ist die Frage, wie viele Stücke man packen kann, ein **Flussproblem**.
  Das Bergsteigen aus Schritt 4 ist einfacher, findet hier aber dieselben Werte.
- **Kleine Cliquen:** Wenn die Stil-Tabelle fast vollständig ist, gibt es sehr wenige maximale Cliquen; wenn sie leer ist, sind die
  Cliquen einzelne Stile oder Paare und die Aufgabe zerfällt in unabhängige kleine Probleme. Schwer wird es bei vielen sich
  überlappenden Cliquen (`paeckchen4` hat 11).
- **Leere Datei:** `paeckchen2.txt` ist auf den BWINF-Seiten leer. Mit den Angaben der Hinweise (3 Sorten, 9 Stile, 32 Stücke; 0 übrig)
  können wir sie nicht nachrechnen.